In [ ]:
# Now let's all the necessary libraries here 

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.plotting import plot_decision_regions

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
import keras 
import keras_tuner as kt

from keras.callbacks import EarlyStopping
from keras.optimizers import Adam,SGD,RMSprop
from keras.models import Sequential
from keras.layers import Input,Activation
from keras.layers import Dropout,Dense,BatchNormalization

In [31]:
df = pd.read_csv('insurance.csv')

In [32]:
df.shape

(1338, 7)

In [33]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [35]:
# This is also one of the famous dataset so we just need to map sex and age into binary and make seperate columns for region usng onc 

In [36]:
df['sex'] = df['sex'].map({
                                'female': 0,
                                'male' : 1
})

In [37]:
df['smoker'] = df['smoker'].map({
                                'no': 0,
                                'yes' : 1
})

In [38]:
df = pd.get_dummies(df,columns=['region'],dtype=int) # OneHotEncode using get_dummmies

In [40]:
df.head() # All set

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,0,1
1,18,1,33.770,1,0,1725.55230,0,0,1,0
2,28,1,33.000,3,0,4449.46200,0,0,1,0
3,33,1,22.705,0,0,21984.47061,0,1,0,0
4,32,1,28.880,0,0,3866.85520,0,1,0,0


In [46]:
# I also want to see the corr of the values with charges
df.corr()['charges'] # We won't remove the non significant columns here for now ; maybe later during experimenting

age                 0.299008
sex                 0.057292
bmi                 0.198341
children            0.067998
smoker              0.787251
charges             1.000000
region_northeast    0.006349
region_northwest   -0.039905
region_southeast    0.073982
region_southwest   -0.043210
Name: charges, dtype: float64

In [41]:
# Now we will split the data into X,y and then training and testing set for our ann

In [42]:
X,y = df.drop(columns=['charges']),df['charges']

In [44]:
X_train , X_test , y_train , y_test = train_test_split(
                                                            X,y,
                                                            test_size = 0.2,
                                                            random_state = 42
)

In [47]:
# Now we have one last step left that is Normalization which we will do using StandardScaler
scale = StandardScaler()
scale.fit(X_train)

scale.transform(X_train)
scale.transform(X_test)

array([[ 0.40114007, -1.0246016 , -0.89153925, ..., -0.56079971,
        -0.59966106, -0.5723141 ],
       [-0.23863782, -1.0246016 , -0.08946143, ...,  1.78316783,
        -0.59966106, -0.5723141 ],
       [ 1.75178229, -1.0246016 , -0.60845296, ...,  1.78316783,
        -0.59966106, -0.5723141 ],
       ...,
       [-0.09646495,  0.97598911, -0.41972876, ..., -0.56079971,
        -0.59966106, -0.5723141 ],
       [ 1.04091797, -1.0246016 ,  2.78941026, ..., -0.56079971,
         1.66760869, -0.5723141 ],
       [ 0.82765867, -1.0246016 ,  0.60252728, ..., -0.56079971,
        -0.59966106,  1.74729228]], shape=(268, 9))

In [48]:
# Now we are readly for the model building phase